In [1]:
import os
import pandas as pd
import datetime
from sqlalchemy import (
    create_engine, 
    URL, 
    MetaData,
    Table, 
    Column, 
    Integer,
    BigInteger,
    Identity,
    TIMESTAMP,
    inspect,
    JSON,
    Boolean,
    Numeric
    )
    
from sqlalchemy.sql import text, quoted_name

In [11]:
# Chargement de la base de données de test
data_path = '..\\..\\data\\original'
test_df = pd.read_csv(os.path.join(data_path, 'demonstration_data.csv'), 
                      sep = ";")

In [12]:
test_df.head()

,SK_ID_CURR,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,...,ORGANIZATION_TYPE_Bank,ORGANIZATION_TYPE_Business Entity Type 3,ORGANIZATION_TYPE_Construction,ORGANIZATION_TYPE_Military,ORGANIZATION_TYPE_Realtor,ORGANIZATION_TYPE_School,ORGANIZATION_TYPE_Security,ORGANIZATION_TYPE_Self-employed,FONDKAPREMONT_MODE_reg oper account,WALLSMATERIAL_MODE_Panel
0,100001,0,135000.0,568800.0,0.018850,-19241,-2329.0,-5170.0,-812,NaN,...,FAUX,FAUX,FAUX,FAUX,FAUX,FAUX,FAUX,FAUX,FAUX,FAUX
1,100005,0,99000.0,222768.0,0.035792,-18064,-4469.0,-9118.0,-1623,NaN,...,FAUX,FAUX,FAUX,FAUX,FAUX,FAUX,FAUX,VRAI,FAUX,FAUX
2,100013,0,202500.0,663264.0,0.019101,-20038,-4458.0,-2175.0,-3503,5.0,...,FAUX,FAUX,FAUX,FAUX,FAUX,FAUX,FAUX,FAUX,FAUX,FAUX
3,100028,2,315000.0,1575000.0,0.026392,-13976,-1866.0,-2000.0,-4208,NaN,...,FAUX,VRAI,FAUX,FAUX,FAUX,FAUX,FAUX,FAUX,VRAI,VRAI
4,100038,1,180000.0,625500.0,0.010032,-13040,-2191.0,-4000.0,-4262,16.0,...,FAUX,VRAI,FAUX,FAUX,FAUX,FAUX,FAUX,FAUX,FAUX,FAUX


In [13]:
# Connection à l'utilisateur admin de PostgresSQL
POSTGRES_PASSWORD = "runJNJsCS3a3drSV"

DATABASE_NAME = "scoring_db"
DATABASE_USER = "scoring_app"
DATABASE_USER_PASSWORD = "scoring_app_pswd"

admin_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password=POSTGRES_PASSWORD,
    host="localhost",
    port=5432,
    database="postgres",
)

admin_engine = create_engine(admin_url, isolation_level="AUTOCOMMIT")

In [14]:
# A travers l'admin, je crée un nouvel utilisateur pour ma base de données de scoring
create_user_sql = text(f"CREATE USER {quoted_name(DATABASE_USER, False)} WITH PASSWORD :database_password")
with admin_engine.connect() as connection:
    connection.execute(create_user_sql,{"database_password": DATABASE_USER_PASSWORD})

In [15]:
# Je crée ensuite la nouvelle base de donnée et j'y attribue l'utilisateur
with admin_engine.connect() as connection:
    connection.execute(text(f"CREATE DATABASE {quoted_name(DATABASE_NAME, False)} OWNER scoring_app"))

In [16]:
# Je peux maintement me connecté à la nouvelle base de données
scoring_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DATABASE_USER,
    password=DATABASE_USER_PASSWORD,
    host="localhost",
    port=5432,
    database=DATABASE_NAME,
)

scoring_engine = create_engine(scoring_url, isolation_level='AUTOCOMMIT')

In [17]:
# J'y ajoute une table avec mes données de tests grâce à la fonction pandas to_sql
test_df.to_sql(name = 'test_clients', con = scoring_engine, if_exists = "replace")

49

In [18]:
# Paramétrage de la clé primaire de la table: l'id des clients
with scoring_engine.connect() as connection:
    connection.execute(text("""
        ALTER TABLE test_clients
        ADD CONSTRAINT clients_pkey PRIMARY KEY ("SK_ID_CURR");
    """))

In [20]:
# Je charge cette table dans un objet MetaData et je crée également dans
# cet objet la table qui sauvegardera les outputs du modèle. 
metadata = MetaData()

test_clients = Table(
    "test_clients",
    metadata,
    autoload_with=scoring_engine,
)

model_logs = Table(
    "model_logs", metadata,
    Column("request_id", BigInteger, Identity(always=True), primary_key=True),
    Column("requested_at", TIMESTAMP, nullable=False, default=datetime.datetime.now(datetime.timezone.utc)),
    Column("requested_params", JSON, nullable=True),
    Column("pred_class", Integer, nullable=False),
    Column("execution_time_ms", Numeric(10, 3), nullable=False),
    Column("error", Boolean, nullable = False),
    Column("error_message", JSON, nullable = True),
)
# Cette commande permet de créer la nouvelle table dans la DB
metadata.create_all(scoring_engine)

In [21]:
#Visualisation de la structure finale de la DB.
inspector = inspect(scoring_engine)

# Liste des tables
tables = inspector.get_table_names()

print("Tables dans la base :")
for table_name in tables:
    print(f"- {table_name}")

    if table_name == "model_logs":
        columns = inspector.get_columns(table_name)

        for column in columns:
            print(
                f"{column['name']} | "
                f"type={column['type']} | "
                f"nullable={column['nullable']} | "
                f"default={column['default']}"
            )

Tables dans la base :
- test_clients
- model_logs
request_id | type=BIGINT | nullable=False | default=None
requested_at | type=TIMESTAMP | nullable=False | default=None
requested_params | type=JSON | nullable=True | default=None
pred_class | type=INTEGER | nullable=False | default=None
execution_time_ms | type=NUMERIC(10, 3) | nullable=False | default=None
error | type=BOOLEAN | nullable=False | default=None
error_message | type=JSON | nullable=True | default=None
